In [ ]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numba
from numba import njit, prange
import os

In [ ]:
# =====
# Array
# =====
alpha_list = np.arange(0, 91, 0.1) 
phi_list = np.zeros(len(alpha_list))
theta_list = np.zeros(len(alpha_list))

P_0_list = np.zeros(len(alpha_list))
P_1_list = np.zeros(len(alpha_list))

# ==========
# Parameters
# ==========

cj = 0.1
dt = 0.01


In [ ]:
# ===========
# Calculation
# ===========

n_points = len(alpha_list)

@njit(cache=True)
def compute_probability(alpha_list, n_points, cj, dt, phi_list, theta_list, P_0_list, P_1_list):
    
    for i in range(n_points):
        # -----------------
        # Angles definition
        # -----------------
        theta_list[i] = np.radians(alpha_list[i])    # z/x angle respect to z axis
        phi_list[i] = theta_list[i] - np.pi/2   # +z/-z angle  
    
        # ------------ 
        # g parameters
        # ------------
        g_z = np.cos(theta_list[i])
        g_x = np.sin(theta_list[i])
        g_0 = np.cos(phi_list[i]/2)
        g_1 = np.sin(phi_list[i]/2)
    
        # -------------
        # Probabilities
        # -------------
        P_0_list[i] = (g_0**2) * (np.cos(cj*dt))**2 + ((g_0 * g_z + g_1 * g_x)**2) * (np.sin(cj*dt))**2
        
        P_1_list[i] = (g_1**2) * (np.cos(cj*dt))**2 + ((g_0 * g_x - g_1 * g_z)**2) * (np.sin(cj*dt))**2

    return P_0_list, P_1_list

In [ ]:
P0, P1 = compute_probability(alpha_list, n_points, cj, dt, phi_list, theta_list, P_0_list, P_1_list)

In [ ]:
path_imm = '/home/francesco/Collisional_Methods/Codes/Results/Images'
save_file = os.path.join(path_imm, "Prob(theta).png")

plt.close('all')

fig01, ax = plt.subplots(figsize = (10,5)) 

ax.plot(alpha_list, P0, label='$P_0$', linewidth=2, color='blue')
ax.plot(alpha_list, P1, label='$P_1$', linewidth=2, color='orange')

ax.set_title(r'Probability as function of $\theta$°')
ax.set_xlabel(r'$\theta°$')
ax.set_ylabel('Probability')
ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
ax.set_ylim(-0.1, 1.1)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()

fig01.savefig(save_file, dpi=300, bbox_inches='tight')

plt.show()